In [49]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
from tabulate import tabulate
from pycoingecko import CoinGeckoAPI
from scipy.stats import skew

class TrendAnalyzer:
    """
    A class to analyze price trends for multiple cryptocurrency assets, with enhanced trend-type analysis,
    efficient recomputation, and historical basket filtering.

    Attributes:
        asset_ids (list): List of asset IDs to analyze
        data_path (str): Path to the directory containing candle data
        btc_data_path (str): Path to the Bitcoin data file
        use_btc_adjusted (bool): Whether to prioritize BTC-adjusted prices in output
        verbose (bool): Whether to print detailed processing information
        lookback_days (int): Lookback period for overall trend persistence and metrics
        trend_metrics_lookback (int or str): Lookback period for per-trend-type metrics ('all' for full data)
        ticker_mapping (dict): Mapping from ticker symbols to asset IDs
        asset_data (dict): Dictionary storing raw data, indicators, classifications, and metrics
        static_computed (bool): Flag to track if static computations are done
    """

    def __init__(self, asset_ids, data_path, btc_data_path, use_btc_adjusted=True, verbose=True, lookback_days=90, trend_metrics_lookback='all'):
        self.asset_ids = asset_ids
        self.data_path = data_path
        self.btc_data_path = btc_data_path
        self.use_btc_adjusted = use_btc_adjusted
        self.verbose = verbose
        self.lookback_days = lookback_days
        self.trend_metrics_lookback = trend_metrics_lookback
        self.ma_periods = {
            'Short Term': [3, 5, 7, 14],
            'Medium Term': [21, 30, 45, 63],
            'Long Term': [84, 100, 120, 150, 200, 252, 365]
        }
        self.ticker_mapping = self._get_ticker_mapping()
        self.asset_data = {}
        self.static_computed = False

    def _get_ticker_mapping(self):
        cg = CoinGeckoAPI()
        coins_list = cg.get_coins_list()
        coins_df = pd.DataFrame(coins_list)
        
        known_mappings = {
            'VIRTUAL': 'virtual-protocol',
            'HYPE': 'hyperliquid',
            'YNE': 'yesnoerror'
        }
        
        mapping = {}
        for ticker, coin_id in known_mappings.items():
            if coin_id in self.asset_ids:
                mapping[ticker] = coin_id
        
        filtered_coins_df = coins_df[coins_df['id'].isin(self.asset_ids)]
        for _, row in filtered_coins_df.iterrows():
            ticker = row['symbol'].upper()
            if ticker not in mapping:
                mapping[ticker] = row['id']
        
        return mapping

    def _load_data(self, asset_id):
        data_path = f"{self.data_path}{asset_id}_candles.csv"
        try:
            data = pd.read_csv(data_path)
        except FileNotFoundError:
            if self.verbose:
                print(f"Warning: File not found for {asset_id}: {data_path}")
            return pd.DataFrame()
        
        data['date'] = pd.to_datetime(data['date'])
        data.dropna(subset=['close'], inplace=True)
        if data.empty:
            if self.verbose:
                print(f"No valid data for {asset_id} after dropping NaN in 'close'.")
            return pd.DataFrame()

        if asset_id != 'bitcoin':
            try:
                btc_data = pd.read_csv(self.btc_data_path)
                btc_data['date'] = pd.to_datetime(btc_data['date'])
                btc_data = btc_data[['date', 'close']].rename(columns={'close': 'btc_close'})
                btc_data.dropna(subset=['btc_close'], inplace=True)
                
                data = data.merge(btc_data, on='date', how='inner')
                if data.empty:
                    if self.verbose:
                        print(f"No overlapping dates between {asset_id} and Bitcoin data after merge.")
                    return pd.DataFrame()

                if 'close' in data.columns and 'btc_close' in data.columns:
                    data[f'{asset_id}_btc'] = data['close'] / data['btc_close']
                else:
                    if self.verbose:
                        print(f"Missing columns after merge for {asset_id}. Available columns: {list(data.columns)}")
                    return pd.DataFrame()
                data = data.drop(columns=['btc_close'], errors='ignore')
            except Exception as e:
                if self.verbose:
                    print(f"Error merging Bitcoin data for {asset_id}: {e}")
                return pd.DataFrame()

        return data

    def _calculate_indicators(self, data, price_column, asset_id, suffix=''):
        if price_column not in data.columns or data.empty:
            if self.verbose:
                print(f"Column '{price_column}' not found for {asset_id} or data is empty. Available columns: {list(data.columns)}")
            return

        data[price_column] = pd.to_numeric(data[price_column], errors='coerce')
        data.dropna(subset=[price_column], inplace=True)
        if data.empty:
            if self.verbose:
                print(f"No valid data for {asset_id} after numeric conversion for {price_column}.")
            return

        for term, periods in self.ma_periods.items():
            for period in periods:
                if len(data) >= period:
                    data[f'EMA_{suffix}_{period}'] = data[price_column].ewm(
                        span=period, min_periods=period, adjust=False
                    ).mean()
                else:
                    data[f'EMA_{suffix}_{period}'] = np.nan

        for term, periods in self.ma_periods.items():
            for period in periods:
                if len(data) >= period:
                    data[f'RoC_{suffix}_{period}'] = (
                        (data[price_column] - data[price_column].shift(period))
                        / data[price_column].shift(period) * 100
                    )
                else:
                    data[f'RoC_{suffix}_{period}'] = np.nan

    def classify_trend(self, mas, rocs, data, suffix):
        n_mas = len(mas.dropna())
        n_rocs = len(rocs.dropna())
        if n_mas == 0 or n_rocs == 0:
            return "Neutral"
        
        pos_mas = 0
        for col in mas.index:
            period = int(col.replace(f'EMA_{suffix}_', ''))
            current_ma = data[f'EMA_{suffix}_{period}'].iloc[-1]
            prev_ma = data[f'EMA_{suffix}_{period}'].iloc[-2] if len(data) > 1 else np.nan
            if not np.isnan(current_ma) and not np.isnan(prev_ma) and current_ma > prev_ma:
                pos_mas += 1
        
        pos_mas = pos_mas / n_mas * 100 if n_mas > 0 else 0
        pos_rocs = sum(1 for roc in rocs if not np.isnan(roc) and roc > 0) / n_rocs * 100 if n_rocs > 0 else 0
        
        avg_pos = (pos_mas + pos_rocs) / 2
        
        if avg_pos >= 75:
            return "Strong Bull"
        elif avg_pos >= 50:
            return "Weak Bull"
        elif avg_pos <= 25:
            return "Strong Bear"
        elif avg_pos <= 40:
            return "Weak Bear"
        return "Neutral"

    def _compute_overall_classification(self, trends):
        valid_trends = [t for t in trends if t not in [None, np.nan, 'N/A']]
        if not valid_trends:
            return "Neutral"
        overall_pos = sum(1 for c in valid_trends if c in ["Strong Bull", "Weak Bull"]) / len(valid_trends) * 100
        if overall_pos >= 75:
            return "Strong Bull"
        elif overall_pos >= 50:
            return "Weak Bull"
        elif overall_pos <= 25:
            return "Strong Bear"
        elif overall_pos <= 40:
            return "Weak Bear"
        return "Neutral"

    def _create_classified_data(self, data, asset_id):
        if data.empty:
            return pd.DataFrame()
        classified_data = data.copy()

        # Compute daily returns for trend-type analysis
        classified_data['returns_usd'] = classified_data['close'].pct_change()
        if asset_id != 'bitcoin' and f'{asset_id}_btc' in classified_data.columns:
            classified_data['returns_btc'] = classified_data[f'{asset_id}_btc'].pct_change()
        else:
            classified_data['returns_btc'] = np.nan

        # USD classifications
        usd_price_column = 'close'
        self._calculate_indicators(classified_data, usd_price_column, asset_id, suffix='USD')
        
        short_term_mas_usd = [f'EMA_USD_{period}' for period in self.ma_periods['Short Term']]
        medium_term_mas_usd = [f'EMA_USD_{period}' for period in self.ma_periods['Medium Term']]
        long_term_mas_usd = [f'EMA_USD_{period}' for period in self.ma_periods['Long Term']]
        
        short_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Short Term']]
        medium_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Medium Term']]
        long_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Long Term']]

        classified_data['Short Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[short_term_mas_usd], row[short_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Medium Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[medium_term_mas_usd], row[medium_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Long Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[long_term_mas_usd], row[long_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Overall (USD)'] = classified_data.apply(
            lambda row: self._compute_overall_classification(
                [row['Short Term (USD)'], row['Medium Term (USD)'], row['Long Term (USD)']]
            ), axis=1
        )

        # BTC classifications
        if asset_id != 'bitcoin' and f'{asset_id}_btc' in classified_data.columns:
            btc_price_column = f'{asset_id}_btc'
            self._calculate_indicators(classified_data, btc_price_column, asset_id, suffix='BTC')
            
            short_term_mas_btc = [f'EMA_BTC_{period}' for period in self.ma_periods['Short Term']]
            medium_term_mas_btc = [f'EMA_BTC_{period}' for period in self.ma_periods['Medium Term']]
            long_term_mas_btc = [f'EMA_BTC_{period}' for period in self.ma_periods['Long Term']]
            
            short_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Short Term']]
            medium_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Medium Term']]
            long_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Long Term']]

            classified_data['Short Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[short_term_mas_btc], row[short_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Medium Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[medium_term_mas_btc], row[medium_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Long Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[long_term_mas_btc], row[long_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Overall (BTC)'] = classified_data.apply(
                lambda row: self._compute_overall_classification(
                    [row['Short Term (BTC)'], row['Medium Term (BTC)'], row['Long Term (BTC)']]
                ), axis=1
            )
        else:
            classified_data['Short Term (BTC)'] = pd.NA
            classified_data['Medium Term (BTC)'] = pd.NA
            classified_data['Long Term (BTC)'] = pd.NA
            classified_data['Overall (BTC)'] = pd.NA
        
        return classified_data

    def calculate_sharpe_sortino(self, data, returns_column):
        if data.empty or len(data) < 2 or returns_column not in data.columns:
            return 0.0, 0.0

        returns = data[returns_column].dropna()
        if len(returns) < 2:
            return 0.0, 0.0

        mean_return = returns.mean() * 365
        std_return = returns.std() * np.sqrt(365)
        sharpe = mean_return / std_return if std_return != 0 else 0.0

        downside_returns = returns[returns < 0]
        downside_deviation = downside_returns.std() * np.sqrt(365) if len(downside_returns) > 0 else 0.0
        sortino = mean_return / downside_deviation if downside_deviation != 0 else 0.0

        return sharpe, sortino

    def calculate_trend_type_metrics(self, data, trend_type, lookback):
        if data.empty or trend_type not in data.columns:
            return {}

        # Apply lookback period
        if lookback != 'all':
            latest_date = data['date'].max()
            start_date = latest_date - timedelta(days=lookback)
            data = data[data['date'] >= start_date].copy()
        if data.empty or len(data) < 2:
            return {}

        # Group by trend type and compute metrics
        metrics = {'USD': {}, 'BTC': {}}
        trend_types = ['Strong Bull', 'Weak Bull', 'Strong Bear', 'Weak Bear', 'Neutral']

        for t in trend_types:
            trend_data = data[data[trend_type] == t].copy()
            if trend_data.empty:
                metrics['USD'][t] = {'sharpe': 0.0, 'sortino': 0.0, 'mean': 0.0, 'median': 0.0, 'skew': 0.0}
                metrics['BTC'][t] = {'sharpe': 0.0, 'sortino': 0.0, 'mean': 0.0, 'median': 0.0, 'skew': 0.0}
                continue

            # USD metrics
            returns_usd = trend_data['returns_usd'].dropna()
            if len(returns_usd) >= 2:
                sharpe_usd, sortino_usd = self.calculate_sharpe_sortino(trend_data, 'returns_usd')
                mean_usd = returns_usd.mean()
                median_usd = returns_usd.median()
                skew_usd = skew(returns_usd, nan_policy='omit')
            else:
                sharpe_usd, sortino_usd, mean_usd, median_usd, skew_usd = 0.0, 0.0, 0.0, 0.0, 0.0
            metrics['USD'][t] = {
                'sharpe': sharpe_usd,
                'sortino': sortino_usd,
                'mean': mean_usd,
                'median': median_usd,
                'skew': skew_usd
            }

            # BTC metrics
            returns_btc = trend_data['returns_btc'].dropna()
            if len(returns_btc) >= 2:
                sharpe_btc, sortino_btc = self.calculate_sharpe_sortino(trend_data, 'returns_btc')
                mean_btc = returns_btc.mean()
                median_btc = returns_btc.median()
                skew_btc = skew(returns_btc, nan_policy='omit')
            else:
                sharpe_btc, sortino_btc, mean_btc, median_btc, skew_btc = 0.0, 0.0, 0.0, 0.0, 0.0
            metrics['BTC'][t] = {
                'sharpe': sharpe_btc,
                'sortino': sortino_btc,
                'mean': mean_btc,
                'median': median_btc,
                'skew': skew_btc
            }

        return metrics

    def calculate_transition_probabilities(self, data, trend_type, lookback_days):
        if data.empty or len(data) < 2:
            return 0.0

        latest_date = data['date'].max()
        start_date = latest_date - timedelta(days=lookback_days)
        data = data[data['date'] >= start_date].copy()
        if len(data) < 2:
            return 0.0

        current_trend = data[trend_type].iloc[-1]
        is_bullish = current_trend in ['Strong Bull', 'Weak Bull']

        prev_trend = data[trend_type].shift(1)
        transitions = prev_trend + " -> " + data[trend_type]
        transitions = transitions.dropna()
        if transitions.empty:
            return 0.0

        total_bullish = len(data[data[trend_type].isin(['Strong Bull', 'Weak Bull'])]) - 1
        if total_bullish <= 0:
            return 0.0

        if is_bullish:
            bull_to_bull = len(transitions[transitions.str.contains('Bull ->.*Bull', na=False)])
            persistence_prob = bull_to_bull / total_bullish if total_bullish > 0 else 0.0
        else:
            bear_to_bear = len(transitions[transitions.str.contains('Bear ->.*Bear', na=False)])
            total_bearish = len(data) - total_bullish - 1
            persistence_prob = bear_to_bear / total_bearish if total_bearish > 0 else 0.0

        return persistence_prob

    def _compute_lookback_metrics(self):
        results = []
        reverse_mapping = {v: k for k, v in self.ticker_mapping.items()}
        
        for asset_id in tqdm(self.asset_ids, desc="Computing lookback metrics", disable=not self.verbose):
            classified_data = self.asset_data.get(asset_id, {}).get('classified_data', pd.DataFrame())
            raw_data = self.asset_data.get(asset_id, {}).get('raw_data', pd.DataFrame())

            latest_classifications = classified_data.iloc[-1].to_dict() if not classified_data.empty else {
                'Short Term (USD)': 'N/A',
                'Medium Term (USD)': 'N/A',
                'Long Term (USD)': 'N/A',
                'Overall (USD)': 'N/A',
                'Short Term (BTC)': 'N/A',
                'Medium Term (BTC)': 'N/A',
                'Long Term (BTC)': 'N/A',
                'Overall (BTC)': 'N/A'
            }

            # Overall Sharpe and Sortino
            sharpe_usd, sortino_usd = self.calculate_sharpe_sortino(raw_data, 'close')
            sharpe_btc, sortino_btc = (self.calculate_sharpe_sortino(raw_data, f'{asset_id}_btc')
                                       if asset_id != 'bitcoin' and f'{asset_id}_btc' in raw_data.columns
                                       else (0.0, 0.0))

            # Overall trend persistence
            trans_prob_usd = self.calculate_transition_probabilities(classified_data, 'Overall (USD)', self.lookback_days)
            trans_prob_btc = (self.calculate_transition_probabilities(classified_data, 'Overall (BTC)', self.lookback_days)
                              if asset_id != 'bitcoin' else 0.0)

            # Per-trend-type metrics
            trend_metrics = {}
            for trend_type in ['Short Term (USD)', 'Medium Term (USD)', 'Long Term (USD)', 'Overall (USD)',
                              'Short Term (BTC)', 'Medium Term (BTC)', 'Long Term (BTC)', 'Overall (BTC)']:
                trend_metrics[trend_type] = self.calculate_trend_type_metrics(classified_data, trend_type, self.trend_metrics_lookback)

            # Update asset_data with trend metrics
            self.asset_data[asset_id]['trend_metrics'] = trend_metrics

            # Get current trend type metrics for summary
            current_trend_usd = latest_classifications.get('Overall (USD)', 'N/A')
            current_trend_btc = latest_classifications.get('Overall (BTC)', 'N/A')
            trend_metrics_usd = trend_metrics['Overall (USD)'].get(current_trend_usd, {'sharpe': 0.0, 'sortino': 0.0, 'mean': 0.0, 'median': 0.0, 'skew': 0.0})
            trend_metrics_btc = trend_metrics['Overall (BTC)'].get(current_trend_btc, {'sharpe': 0.0, 'sortino': 0.0, 'mean': 0.0, 'median': 0.0, 'skew': 0.0})

            ticker = reverse_mapping.get(asset_id, asset_id.upper())
            results.append({
                'Ticker': ticker,
                'Short Term Trend (USD)': latest_classifications.get('Short Term (USD)', 'N/A'),
                'Medium Term Trend (USD)': latest_classifications.get('Medium Term (USD)', 'N/A'),
                'Long Term Trend (USD)': latest_classifications.get('Long Term (USD)', 'N/A'),
                'Overall Trend (USD)': latest_classifications.get('Overall (USD)', 'N/A'),
                'Short Term Trend (BTC)': latest_classifications.get('Short Term (BTC)', 'N/A'),
                'Medium Term Trend (BTC)': latest_classifications.get('Medium Term (BTC)', 'N/A'),
                'Long Term Trend (BTC)': latest_classifications.get('Long Term (BTC)', 'N/A'),
                'Overall Trend (BTC)': latest_classifications.get('Overall (BTC)', 'N/A'),
                'Sharpe Ratio (USD)': sharpe_usd,
                'Sortino Ratio (USD)': sortino_usd,
                'Sharpe Ratio (BTC)': sharpe_btc,
                'Sortino Ratio (BTC)': sortino_btc,
                'Trend Persistence (USD)': trans_prob_usd,
                'Trend Persistence (BTC)': trans_prob_btc,
                'Current Trend Sharpe (USD)': trend_metrics_usd['sharpe'],
                'Current Trend Sortino (USD)': trend_metrics_usd['sortino'],
                'Current Trend Mean Return (USD)': trend_metrics_usd['mean'],
                'Current Trend Median Return (USD)': trend_metrics_usd['median'],
                'Current Trend Skew (USD)': trend_metrics_usd['skew'],
                'Current Trend Sharpe (BTC)': trend_metrics_btc['sharpe'],
                'Current Trend Sortino (BTC)': trend_metrics_btc['sortino'],
                'Current Trend Mean Return (BTC)': trend_metrics_btc['mean'],
                'Current Trend Median Return (BTC)': trend_metrics_btc['median'],
                'Current Trend Skew (BTC)': trend_metrics_btc['skew'],
                'Latest Date': classified_data['date'].iloc[-1].date() if not classified_data.empty else None
            })

        return pd.DataFrame(results)

    def analyze_multiple_assets(self):
        # Static computations
        self.asset_data = {}
        for asset_id in tqdm(self.asset_ids, desc="Analyzing assets (static)", disable=not self.verbose):
            raw_data = self._load_data(asset_id)
            classified_data = self._create_classified_data(raw_data, asset_id)
            self.asset_data[asset_id] = {
                'raw_data': raw_data.copy(),
                'classified_data': classified_data.copy(),
                'price_column_usd': 'close',
                'price_column_btc': f'{asset_id}_btc' if asset_id != 'bitcoin' and f'{asset_id}_btc' in classified_data.columns else None
            }
        self.static_computed = True

        # Dynamic computations
        summary_df = self._compute_lookback_metrics()
        
        if self.verbose:
            print("\nMost Recent Trend Analysis Summary for Multiple Assets:")
            print(tabulate(summary_df, headers='keys', tablefmt='pretty', showindex=False))
        
        return summary_df

    def recompute_lookback_metrics(self, lookback_days=None, trend_metrics_lookback=None):
        if not self.static_computed:
            raise ValueError("Static computations must be performed first by calling analyze_multiple_assets().")

        if lookback_days is not None:
            self.lookback_days = lookback_days
        if trend_metrics_lookback is not None:
            self.trend_metrics_lookback = trend_metrics_lookback

        summary_df = self._compute_lookback_metrics()
        
        if self.verbose:
            print("\nUpdated Trend Analysis Summary with New Lookback Parameters:")
            print(tabulate(summary_df, headers='keys', tablefmt='pretty', showindex=False))
        
        return summary_df

    def display_trend_type_metrics(self, asset_id=None):
        if not self.static_computed:
            raise ValueError("Static computations must be performed first by calling analyze_multiple_assets().")

        if asset_id:
            assets = [asset_id]
        else:
            assets = self.asset_ids

        for asset_id in assets:
            if asset_id not in self.asset_data:
                print(f"No data available for {asset_id}.")
                continue
            trend_metrics = self.asset_data[asset_id].get('trend_metrics', {})
            print(f"\nTrend-Type Metrics for {asset_id}:")
            for trend_type, metrics in trend_metrics.items():
                print(f"\n{trend_type}:")
                print("USD Metrics:")
                for t, m in metrics['USD'].items():
                    print(f"  {t}:")
                    for k, v in m.items():
                        print(f"    {k}: {v:.6f}")
                print("BTC Metrics:")
                for t, m in metrics['BTC'].items():
                    print(f"  {t}:")
                    for k, v in m.items():
                        print(f"    {k}: {v:.6f}")

    def filter_basket_historical(self, dates):
        if not self.static_computed:
            raise ValueError("Static computations must be performed first by calling analyze_multiple_assets().")

        historical_baskets = []
        for date in dates:
            date = pd.to_datetime(date)
            basket = []
            for asset_id in self.asset_ids:
                classified_data = self.asset_data.get(asset_id, {}).get('classified_data', pd.DataFrame())
                if classified_data.empty or classified_data['date'].max() < date:
                    continue

                # Get data up to the specified date
                data = classified_data[classified_data['date'] <= date].copy()
                if data.empty:
                    continue

                # Get the latest classifications for this date
                latest_classifications = data.iloc[-1].to_dict()

                # Apply filtering criteria
                if (latest_classifications.get('Short Term (USD)', 'N/A') in ['Weak Bull', 'Strong Bull'] and
                    latest_classifications.get('Short Term (BTC)', 'N/A') in ['Weak Bull', 'Strong Bull'] and
                    (latest_classifications.get('Medium Term (BTC)', 'N/A') in ['Weak Bull', 'Strong Bull'] or
                     latest_classifications.get('Medium Term (USD)', 'N/A') in ['Weak Bull', 'Strong Bull']) and
                    latest_classifications.get('Overall (BTC)', 'N/A') in ['Strong Bull']):
                    ticker = [k for k, v in self.ticker_mapping.items() if v == asset_id][0]
                    basket.append({
                        'Date': date.date(),
                        'Ticker': ticker,
                        'Short Term Trend (USD)': latest_classifications.get('Short Term (USD)', 'N/A'),
                        'Medium Term Trend (USD)': latest_classifications.get('Medium Term (USD)', 'N/A'),
                        'Long Term Trend (USD)': latest_classifications.get('Long Term (USD)', 'N/A'),
                        'Overall Trend (USD)': latest_classifications.get('Overall (USD)', 'N/A'),
                        'Short Term Trend (BTC)': latest_classifications.get('Short Term (BTC)', 'N/A'),
                        'Medium Term Trend (BTC)': latest_classifications.get('Medium Term (BTC)', 'N/A'),
                        'Long Term Trend (BTC)': latest_classifications.get('Long Term (BTC)', 'N/A'),
                        'Overall Trend (BTC)': latest_classifications.get('Overall (BTC)', 'N/A')
                    })

            historical_baskets.extend(basket)

        basket_df = pd.DataFrame(historical_baskets)
        if basket_df.empty:
            print("No assets met the filtering criteria for the specified dates.")
        else:
            print("\nHistorical Basket Analysis:")
            print(tabulate(basket_df, headers='keys', tablefmt='pretty', showindex=False))
        return basket_df

# Usage
if __name__ == "__main__":
    from scripts.assetsRoster import carteira_AC, carteira_HB, carteira_LC, carteira_EXC, others

    all_assets = list(set(carteira_AC + carteira_HB + carteira_LC + carteira_EXC + others))
    
    analyzer = TrendAnalyzer(
        asset_ids=all_assets,
        data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/',
        btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv',
        use_btc_adjusted=False,
        verbose=True,
        lookback_days=365,
        trend_metrics_lookback=365
    )
    summary_df = analyzer.analyze_multiple_assets()

    

Computing lookback metrics: 100%|██████████| 128/128 [00:03<00:00, 35.40it/s]


Most Recent Trend Analysis Summary for Multiple Assets:
+----------+------------------------+-------------------------+-----------------------+---------------------+------------------------+-------------------------+-----------------------+---------------------+--------------------+---------------------+--------------------+---------------------+-------------------------+-------------------------+----------------------------+-----------------------------+---------------------------------+-----------------------------------+--------------------------+----------------------------+-----------------------------+---------------------------------+-----------------------------------+--------------------------+-------------+
|  Ticker  | Short Term Trend (USD) | Medium Term Trend (USD) | Long Term Trend (USD) | Overall Trend (USD) | Short Term Trend (BTC) | Medium Term Trend (BTC) | Long Term Trend (BTC) | Overall Trend (BTC) | Sharpe Ratio (USD) | Sortino Ratio (USD) | Sharpe Ratio (BTC) | S

In [ ]:
# Display trend-type metrics for a specific asset
analyzer.display_trend_type_metrics(asset_id='ethereum')



In [ ]:
# Recompute lookback metrics with a different lookback period
summary_df = analyzer.recompute_lookback_metrics(lookback_days=30, trend_metrics_lookback=180)



In [50]:
# Historical basket analysis
dates = pd.date_range(start='2024-01-01', end='2025-03-18', freq='d')  # Weekly intervals
historical_basket = analyzer.filter_basket_historical(dates) 

# should be able to stablish filters for the basket at input level 



Historical Basket Analysis:
+------------+----------+------------------------+-------------------------+-----------------------+---------------------+------------------------+-------------------------+-----------------------+---------------------+
|    Date    |  Ticker  | Short Term Trend (USD) | Medium Term Trend (USD) | Long Term Trend (USD) | Overall Trend (USD) | Short Term Trend (BTC) | Medium Term Trend (BTC) | Long Term Trend (BTC) | Overall Trend (BTC) |
+------------+----------+------------------------+-------------------------+-----------------------+---------------------+------------------------+-------------------------+-----------------------+---------------------+
| 2024-01-01 |   IMX    |       Weak Bull        |        Weak Bull        |       Weak Bull       |     Strong Bull     |       Weak Bull        |        Weak Bull        |       Weak Bull       |     Strong Bull     |
| 2024-01-01 |   AKT    |      Strong Bull       |        Weak Bull        |       Weak Bul

In [36]:
summary_df[(summary_df['Short Term Trend (USD)'].isin(['Weak Bull', 'Strong Bull'])) &
           (summary_df['Short Term Trend (BTC)'].isin(['Weak Bull', 'Strong Bull'])) &
            
           ((summary_df['Medium Term Trend (BTC)'].isin(['Weak Bull', 'Strong Bull'])) |
           (summary_df['Medium Term Trend (USD)'].isin(['Weak Bull', 'Strong Bull']))) &
           #(summary_df['Long Term Trend (USD)'].isin(['Strong Bull'])) &
           (summary_df['Overall Trend (BTC)'].isin(['Strong Bull']))
           ]

,Ticker,Short Term Trend (USD),Medium Term Trend (USD),Long Term Trend (USD),Overall Trend (USD),Short Term Trend (BTC),Medium Term Trend (BTC),Long Term Trend (BTC),Overall Trend (BTC),Sharpe Ratio (USD),Sortino Ratio (USD),Sharpe Ratio (BTC),Sortino Ratio (BTC),Trend Persistence (USD),Trend Persistence (BTC),Latest Date
26,USDC,Weak Bull,Strong Bear,Weak Bear,Weak Bear,Strong Bull,Strong Bull,Weak Bull,Strong Bull,0.410582,0.431865,1.781030,2.689778,0.929412,1.0,2025-03-18
28,PAXG,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Weak Bull,Strong Bull,4.630302,7.355734,2.916636,4.443575,1.000000,1.0,2025-03-18
49,XRP,Strong Bull,Strong Bear,Strong Bull,Weak Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,0.614504,1.020962,1.234413,2.349228,1.000000,1.0,2025-03-18
70,XMR,Strong Bull,Strong Bear,Strong Bull,Weak Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,0.290461,0.399269,1.478498,2.329940,0.962500,1.0,2025-03-18
100,OM,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,2.526876,7.311435,2.924207,8.107128,1.000000,1.0,2025-03-18
106,BNB,Strong Bull,Weak Bull,Weak Bull,Strong Bull,Strong Bull,Strong Bull,Weak Bull,Strong Bull,-0.340498,-0.492009,0.963276,1.776534,1.000000,1.0,2025-03-18
110,CAKE,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,0.423789,0.908246,0.926051,2.197961,1.000000,1.0,2025-03-18
125,TUSD,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Strong Bull,Weak Bull,Strong Bull,-0.010583,-0.013115,1.780158,2.775405,1.000000,1.0,2025-03-18
